In [1]:
import sksurv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import pandas as pd
import numpy as np
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import os
import kagglehub
from dataclasses import asdict

from Utils.Config import Config, TrialResult
from Utils.utils import (
    set_seed,
    make_surv_y,
    make_oof_predictions,
    create_features,
    load_config_yaml,
    make_corr_matrix,
    HORIZONS
)

In [2]:
storage = JournalStorage(
    JournalFileBackend("gbsa_journal.log")
)

study = optuna.load_study(
    study_name="gbsa_survival",
    storage=storage,
)

In [3]:
best_trial = study.best_trial
best_trial_num = best_trial.number
print("Best trial:")
print(f"  Number: {best_trial.number}")
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Number: 106
  Value: 0.9629674137093398
  Params: 
    n_estimators: 417
    learning_rate: 0.11063731185373184
    subsample: 0.5905037758154414
    max_depth: 6
    max_features: sqrt
    max_leaf_nodes: 31
    min_samples_split: 4
    min_samples_leaf: 8
    min_impurity_decrease: 0.02904783931441647
    ccp_alpha: 0.004992133979210339
    dropout_rate: 0.0007471276215608402
    validation_fraction: 0.1897124693474042
    n_iter_no_change: None
    tol: 0.0004912572712305443


In [4]:
trial_dir = os.path.join("Trials", "GBSA")
best_trial_path = os.path.join(trial_dir, f'trial_{best_trial.number}', 'config.yaml')
print(f"Best trial config path: {best_trial_path}")
print(f"Best trial config exists: {os.path.exists(best_trial_path)}")

Best trial config path: Trials/GBSA/trial_106/config.yaml
Best trial config exists: True


In [5]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

In [6]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_processed = create_features(train_df)
test_processed = create_features(test_df)

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

event_horizon = np.array(HORIZONS).copy()
event_horizon[-1] = min(event_horizon[-1], train_processed['time_to_hit_hours'].max() - 1e-6)

print("Event horizons:", event_horizon)

Event horizons: [12.         24.         48.         66.99447313]


In [7]:
config = load_config_yaml(best_trial_path)
print("Loaded config:")
print(config)

Loaded config:
Config(seed=42, cv_n_splits=5, cv_n_repeats=10, gbsa_config=GBSAConfig(loss='coxph', n_estimators=417, learning_rate=0.11063731185373184, subsample=0.5905037758154414, max_depth=6, max_features='sqrt', max_leaf_nodes=31, min_samples_split=4, min_samples_leaf=8, min_weight_fraction_leaf=0.0, min_impurity_decrease=0.02904783931441647, criterion='friedman_mse', ccp_alpha=0.004992133979210339, dropout_rate=0.0007471276215608402, validation_fraction=0.1897124693474042, n_iter_no_change=None, tol=0.0004912572712305443, random_state=42, warm_start=False, verbose=0), preprocessing_config=PreprocessingConfig(eps=1e-06, min_speed=0.01, max_hours=9999.0))


In [8]:
seed = config.seed
set_seed(seed)
model_params = config.gbsa_config

n_splits = config.cv_n_splits
n_repeats = config.cv_n_repeats

print(n_splits, n_repeats)

5 10


In [9]:
model = GradientBoostingSurvivalAnalysis(**asdict(model_params))

oof_pred = make_oof_predictions(
    model=model,
    data=train_processed,
    horizons=event_horizon,
    seed=seed,
    n_splits=n_splits,
    n_repeats=n_repeats
)

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
repeat_results = oof_pred['repeat_oof_results']
pred_matrix = []
print("Repeat OOF results structure:")
print("Number of repeats:", len(repeat_results))
print("Keys in first repeat result:", repeat_results[0].keys())
print("Number of folds per repeat:", len(repeat_results[0]['oof_risk']))
print("Shape of OOF predictions for first fold of first repeat:", repeat_results[0]['oof_surv'].shape)

for oof in repeat_results:
    pred_matrix.append(oof['oof_surv'])


oof_pred_matrix = np.stack(pred_matrix, axis=0)
print("Result: ", oof_pred_matrix[0, 0, :5])  # Print first 5 predictions for the first fold of the first repeat
print("OOF prediction matrix shape:", oof_pred_matrix.shape)

Repeat OOF results structure:
Number of repeats: 10
Keys in first repeat result: dict_keys(['repeat', 'oof_risk', 'oof_surv', 'oof_count'])
Number of folds per repeat: 221
Shape of OOF predictions for first fold of first repeat: (221, 4)
Result:  [0.97105441 0.9281472  0.9199504  0.8003355 ]
OOF prediction matrix shape: (10, 221, 4)


In [ ]:
oof_hit_pred_matrix = 1 - oof_pred_matrix # Assuming the first column corresponds to the first horizon
corr = make_corr_matrix(oof_pred_matrix, flatten=True)
print("Correlation matrix shape:", corr.shape)

Correlation matrix shape: (10, 10)


In [ ]:
print("Correlation matrix:")
print(corr)

Correlation matrix:
[[1.         0.99560952 0.99622273 0.99491814 0.99682301 0.99544095
  0.99583586 0.99620427 0.99619009 0.99571774]
 [0.99560952 1.         0.99457213 0.99342775 0.99543402 0.99524879
  0.99428764 0.99563836 0.99477946 0.99554844]
 [0.99622273 0.99457213 1.         0.99408359 0.99581754 0.9963234
  0.99639322 0.99561939 0.99629512 0.99552667]
 [0.99491814 0.99342775 0.99408359 1.         0.99515357 0.99505094
  0.99474202 0.99405607 0.99492288 0.99530661]
 [0.99682301 0.99543402 0.99581754 0.99515357 1.         0.99596984
  0.99701546 0.99656309 0.99672909 0.99624826]
 [0.99544095 0.99524879 0.9963234  0.99505094 0.99596984 1.
  0.99459961 0.99593418 0.99560794 0.99646927]
 [0.99583586 0.99428764 0.99639322 0.99474202 0.99701546 0.99459961
  1.         0.99539815 0.99632987 0.99580577]
 [0.99620427 0.99563836 0.99561939 0.99405607 0.99656309 0.99593418
  0.99539815 1.         0.99613369 0.99569144]
 [0.99619009 0.99477946 0.99629512 0.99492288 0.99672909 0.99560794
 

In [ ]:
print("Mean Correlation:", np.mean(corr[np.triu_indices_from(corr, k=1)]))
print("Max Correlation:", np.max(corr[np.triu_indices_from(corr, k=1)]))
print("Min Correlation:", np.min(corr[np.triu_indices_from(corr, k=1)]))

Mean Correlation: 0.9955953493154021
Max Correlation: 0.9970154615818744
Min Correlation: 0.9934277502522456


# Model Ensemble (differnt Config)

In [ ]:
from Utils.utils import save_top_trial_oofs
save_top_trial_oofs(
    study=study,
    data=train_processed,
    horizons=event_horizon,
    top_ratio=0.3,
    seed=seed,
    n_splits=n_splits,
    n_repeats=n_repeats,
    out_dir="Top OOF"
)

Saving OOF predictions for top 90 trials to Top OOF/top_30_oof_results.json...
Best trial value: 0.9629674137093398
Best Tral Number: 106


KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

...

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]